## Goals: Training the *Final* Models

This notebook trains the model on the full *baseline_dataset* for the final prediction on evaluation data.

Here, we train a model designed to generalize across water stations in Brazil and France. However, you are not required to follow this approach and may opt to train separate models for different geographic *regions*.

This baseline model training example utilizes all available features, with hyperparameters chosen for quick execution rather than optimization. For hyperparameter tuning and feature selection explorations, refer to the `02_exploration` folder.

> **Note:** This notebook requires outputs from the `00 Preprocessing` notebooks.

<img src="../images/notebook-3.png" alt="Experiment Diagram" style="width:75%; text-align:center;" />

### 1. Data Import and Setup

This section imports the necessary libraries, sets up environment paths, and includes custom utility functions.

In [ ]:
import os
import sys
import time as t
import warnings
import logging

# Suppress codecarbon warnings for Apple M2
logging.getLogger("codecarbon").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*codecarbon.*")
warnings.filterwarnings("ignore", category=UserWarning, module="codecarbon")

import matplotlib.pyplot as plt
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import tensorflow as tf

from interpret.glassbox import ExplainableBoostingRegressor
from mapie.regression import MapieQuantileRegressor
from quantile_forest import RandomForestQuantileRegressor
from codecarbon import EmissionsTracker
import platform

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

from src.utils.model import split_dataset, compare_models_per_station, create_deep_model

IS_MAC_M2 = platform.system() == "Darwin" and platform.processor() == "arm"
if IS_MAC_M2:
    print("Running on Apple Silicon - Using TDP estimation for carbon tracking")

logging.getLogger("codecarbon").setLevel(logging.ERROR)

# Initialize results storage
carbon_results = {
    "model": [],
    "week": [],
    "duration_seconds": [],
    "emissions_kg": [],
    "cpu_energy_kwh": []
}

EVAL_DIR = "../../../data/evaluation/"

##### Constants :
- **INPUT_DIR**: Directory for input data (same as in "02 - Feature Engineering").
- **MODEL_DIR**: Directory where trained models are saved.
- **DATASET_DIR**: Directory where the Zenodo dataset is unzipped.

##### Model Parameters

- **SEED**: 42 (for reproducibility)
- **NUMBER_OF_WEEK**: 3 (one model is trained per week)

##### FINAL_MODELS

- **mapie**: Combines LightGBM with MAPIE. **MAPIE** (Model Agnostic Prediction Interval Estimator) computes prediction intervals for any regression model using conformal methods.
- **qrf**: Quantile Random Forest (natively produces prediction intervals)
- **ebm**: Explainable Boosting Machine is used as a exemple that does not natively implement prediction intervals, but that can be customised to do so.

In [ ]:
INPUT_DIR = "../../../data/input/"
MODEL_DIR = "../../../models/"
DATASET_DIR = "../../../dataset/"

SEED = 42
NUMBER_OF_WEEK = 3 # Number of weeks to predict one model is trained per week

FINAL_MODELS = [
                "mapie",
                "qrf",
                ]
mapie_enbpi = {}
mapie = {}
qrf = {}
mapie_aci = {}

COLUMNS_TO_DROP = ["water_flow_week1", "water_flow_week2", "water_flow_week3", "water_flow_week4"]


### 2. Data Loading
Load in the baseline datasets, create the directory to save models.

In [ ]:
dataset_train = pd.read_csv(f"{INPUT_DIR}dataset_baseline.csv")

dataset_train = dataset_train.set_index("ObsDate")

if not os.path.exists(f"{MODEL_DIR}final/"):
    os.makedirs(f"{MODEL_DIR}final/")

Data pre-processing removal of unnecessary columns, setup of the target

In [ ]:
X_train = dataset_train.drop(columns=COLUMNS_TO_DROP)
y_train = {}
for i in range(0, NUMBER_OF_WEEK):
    y_train[i] = dataset_train[f"water_flow_week{i+1}"]


### 2. Models training
#### a. LGBM + MAPIE

## Mapie Model Training Overview

- **Configuration:**  
  - Sets `ALPHA` (0.1) as the prediction interval level.
  - Defines `TIME_VALIDATION` as a split point for creating a validation set.
  - Configures LightGBM parameters (`LGBM_PARAMS`) for quantile regression.




In [ ]:
ALPHA = 0.1
TIME_VALIDATION = "2000-01-01 00:00:00"

LGBM_PARAMS = {
    "max_depth": 15,
    "learning_rate": 0.01,
    "n_estimators": 500,
    "colsample_bytree": 0.7,
    "objective": "quantile",
    "alpha": ALPHA
}

- **Data Preparation:**  
  - Splits `dataset_train` into training and validation subsets using `split_dataset`.
  - Removes unnecessary columns from both the training and validation datasets.
  - Extracts target variables for each week (from `water_flow_week1` to `water_flow_week4`).

- **Model Training:**  
  For each week:
  - Initializes a LightGBM regressor with the specified parameters.
  - Wraps it in a `MapieQuantileRegressor` to estimate prediction intervals.
  - Trains the model on the training data and calibrates it using the validation data.
  - Saves the trained model 

In [ ]:
if "mapie" in FINAL_MODELS: 
    print("Training Mapie")


    train_mapie, val_mapie, val_temporal  = split_dataset(dataset_train, 0.75, TIME_VALIDATION)    

    X_train_mapie = train_mapie.drop(columns=COLUMNS_TO_DROP)
    X_train_mapie = X_train_mapie.drop(columns=["station_code"])
    print(len(X_train_mapie.columns))
    y_train_mapie = {}
    for i in range(0, NUMBER_OF_WEEK):
        y_train_mapie[i] = train_mapie[f"water_flow_week{i+1}"]

    X_val = val_mapie.drop(columns=COLUMNS_TO_DROP)
    X_val = X_val.drop(columns=["station_code"])
    y_val = {}
    y_val[0] = val_mapie["water_flow_week1"]
    for i in range(1, NUMBER_OF_WEEK):
        y_val[i] = val_mapie[f"water_flow_week{i+1}"]



    for i in range(NUMBER_OF_WEEK):
        print(f"Training week {i}")
            # Start carbon tracking
        tracker = EmissionsTracker(
            project_name=f"MAPIE_week_{i}",
            measure_power_secs=1,
            tracking_mode="process",
            log_level="WARNING", 
            save_to_file=False
        )
        tracker.start()
        start_time = t.time()

        # Initialize and train MapieQuantileRegressor
        regressor = lgb.LGBMRegressor(**LGBM_PARAMS)
        mapie[i] = MapieQuantileRegressor(estimator=regressor, method="quantile", cv="split", alpha=ALPHA)
        mapie[i].fit(X_train_mapie, y_train_mapie[i], X_calib=X_val, y_calib=y_val[i])

        emissions_kg = tracker.stop()
        duration = t.time() - start_time
        
        # Store results
        carbon_results["model"].append("MAPIE")
        carbon_results["week"].append(i)
        carbon_results["duration_seconds"].append(duration)
        carbon_results["emissions_kg"].append(emissions_kg)
        # For energy, we can estimate from emissions and duration
        carbon_results["cpu_energy_kwh"].append(emissions_kg * 2.0)        
        # save model with date
        time = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")

        model_path = f"{MODEL_DIR}final/mapie_quantile_{time}_week_{i}.pkl"
        joblib.dump(mapie[i], model_path)


#### b. QRF

- **Training:**  
  Initializes a `RandomForestQuantileRegressor` with the following parameters:
  - 100 estimators
  - Maximum depth of 10
  - Minimum of 10 samples per leaf

  These parameters allow for relatively fast training, though they are not optimized for peak performance. 
  
  The model is then fitted using `X_train` and the corresponding weekly target `y_train[i]`.

In [ ]:
X_train_qrf = X_train.drop(columns=["station_code"])

if "qrf" in FINAL_MODELS:
    for i in range(NUMBER_OF_WEEK):
        print(f"Training week {i}")

        tracker = EmissionsTracker(
            project_name=f"QRF_week_{i}",
            measure_power_secs=1,
            tracking_mode="process",
            log_level="WARNING"
        )
        tracker.start()
        start_time = t.time()

        qrf[i] = RandomForestQuantileRegressor(n_estimators=100, max_depth=5, min_samples_leaf=5)
        # Better at : max_depth=10, min_samples_leaf=10
        qrf[i].fit(X_train_qrf, y_train[i])


        emissions_kg = tracker.stop()
        duration = t.time() - start_time
        
        # Store results
        carbon_results["model"].append("QRF")
        carbon_results["week"].append(i)
        carbon_results["duration_seconds"].append(duration)
        carbon_results["emissions_kg"].append(emissions_kg)
        # For energy, we can estimate from emissions and duration
        carbon_results["cpu_energy_kwh"].append(emissions_kg * 2.0) 

        time = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")
        model_path = f"{MODEL_DIR}final/qrf_100_quantile_{time}_week_{i}.pkl"
        joblib.dump(qrf[i], model_path)

In [ ]:
# Display carbon footprint results
carbon_df = pd.DataFrame(carbon_results)
carbon_summary = carbon_df.groupby("model").agg({
    "duration_seconds": ["mean", "sum"],
    "emissions_kg": ["mean", "sum"],
    "cpu_energy_kwh": ["mean", "sum"]
}).round(6)

print("\n=== COMPUTATIONAL COST SUMMARY ===")
print(carbon_summary)

# Save for paper
carbon_df.to_csv(f"{MODEL_DIR}carbon_footprint.csv", index=False)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Training time by model
carbon_df.groupby("model")["duration_seconds"].sum().plot(kind="bar", ax=axes[0])
axes[0].set_title("Total Training Time (seconds)")
axes[0].set_ylabel("Time (s)")

# Energy consumption
carbon_df.groupby("model")["cpu_energy_kwh"].sum().plot(kind="bar", ax=axes[1])
axes[1].set_title("Total Energy Consumption")
axes[1].set_ylabel("kWh")

plt.tight_layout()
plt.savefig(f"{MODEL_DIR}computational_cost.png", dpi=300, bbox_inches='tight')
plt.show()

### 3. Performance Evaluation on the Full Training Set

> **Note:**  
> The performance displayed here is calculated on the training set. This does not necessarily reflect the models' performance on unseen data.


In [ ]:
y_train_stations = dataset_train["station_code"].values

X_train_eval2 = X_train.copy()
X_train_eval = X_train.drop(columns=["station_code"])
for i in range(NUMBER_OF_WEEK):
    predictions = []
    baseline_day_before = dataset_train["water_flow_lag_1w"]
    predictions.append({"model": "Week before", "prediction": baseline_day_before, "dataset":"train", "stations": y_train_stations, "prediction_interval": None})
    if "mapie" in FINAL_MODELS:
        y_pred_mapie, y_pis_mapie = mapie[i].predict(X_train_eval)
        predictions.append({"model": "LGBM+MAPIE",
                            "prediction": y_pred_mapie,
                            "dataset":"train",
                            "stations": y_train_stations,
                            "prediction_interval": y_pis_mapie})
    if "qrf" in FINAL_MODELS:
        y_pred_qrf = qrf[i].predict(X_train_eval, quantiles="mean", aggregate_leaves_first=False)
        y_pis_qrf = qrf[i].predict(X_train_eval, quantiles=[ALPHA/2, 1-ALPHA/2])
        predictions.append({"model": "QRF",
                            "prediction": y_pred_qrf,
                            "dataset":"train",
                            "stations": y_train_stations,
                            "prediction_interval": y_pis_qrf})
        
    compare_models_per_station(
        y_train[i].values,
        predictions,
        y_train_stations,
        column_to_display="log_likelihood" ,
        title = f"WEEK {i}")

### 4. Coverage on the Full Training Set

> **Note:**  
> The performance displayed here is calculated on the training set. This does not necessarily reflect the models' performance on unseen data.


In [ ]:
for i in range(NUMBER_OF_WEEK):

    baseline_day_before = dataset_train["water_flow_lag_1w"]
    if "mapie" in FINAL_MODELS:
        y_pred_mapie, y_pis_mapie = mapie[i].predict(X_train_eval)
        coverage = (y_train[i].values >= y_pis_mapie[:,0,0]) & (y_train[i].values <= y_pis_mapie[:,1,0])
        print(f"MAPIE coverage of the prediction interval for week {i}: {coverage.mean()}")
    if "qrf" in FINAL_MODELS:
        y_pred_qrf = qrf[i].predict(X_train_eval, quantiles="mean", aggregate_leaves_first=False)
        y_pis_qrf = qrf[i].predict(X_train_eval, quantiles=[ALPHA/2, 1-ALPHA/2])
        coverage = (y_train[i].values >= y_pis_qrf[:,0]) & (y_train[i].values <= y_pis_qrf[:,1])
        print(f"QRF coverage of the prediction interval for week {i}: {coverage.mean()}")